# Rearc Data Quest – Part 3: Analytics
Loads BLS time-series and US population data, then answers the three
analytical questions using Pandas.

In [1]:
import io
import json
import boto3
import pandas as pd

S3_BUCKET        = "kumarvik-data-quest-bucket"
BLS_KEY          = 'bls/pr/pr.data.0.Current'
POPULATION_KEY   = 'population/us_population.json'


## Load data from S3

In [2]:
s3 = boto3.client('s3')

# --- BLS time-series (tab-separated) ---
obj = s3.get_object(Bucket=S3_BUCKET, Key=BLS_KEY)
bls_df = pd.read_csv(io.BytesIO(obj['Body'].read()), sep='\t')

# Trim whitespace from column names (data-cleaning hint from README)
bls_df.rename(columns=lambda c: c.strip(), inplace=True)

# Trim whitespace from all string columns (data-cleaning hint from README)
str_cols = bls_df.select_dtypes('object').columns
bls_df[str_cols] = bls_df[str_cols].apply(lambda c: c.str.strip())
bls_df['value'] = pd.to_numeric(bls_df['value'], errors='coerce')

print('BLS shape:', bls_df.shape)
bls_df.head()


BLS shape: (38232, 5)


,series_id,year,period,value,footnote_codes
0,PRS30006011,1995,Q01,2.6,NaN
1,PRS30006011,1995,Q02,2.1,NaN
2,PRS30006011,1995,Q03,0.9,NaN
3,PRS30006011,1995,Q04,0.1,NaN
4,PRS30006011,1995,Q05,1.4,NaN


In [3]:
# --- Population JSON ---
obj = s3.get_object(Bucket=S3_BUCKET, Key=POPULATION_KEY)
pop_json = json.loads(obj['Body'].read())
pop_df = pd.DataFrame(pop_json['data'])

print('Population shape:', pop_df.shape)
pop_df.head()


Population shape: (11, 4)


,Nation ID,Nation,Year,Population
0,01000US,United States,2013,316128839.0
1,01000US,United States,2014,318857056.0
2,01000US,United States,2015,321418821.0
3,01000US,United States,2016,323127515.0
4,01000US,United States,2017,325719178.0


## Question 1 – Mean & Std of US Population (2013-2018 inclusive)

In [4]:
pop_filtered = pop_df[
    pop_df['Year'].between(2013, 2018)
].copy()

mean_pop = pop_filtered['Population'].mean()
std_pop  = pop_filtered['Population'].std()

print(f'Years included : {sorted(pop_filtered["Year"].unique())}')
print(f'Mean population: {mean_pop:,.0f}')
print(f'Std  population: {std_pop:,.0f}')


Years included : [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Mean population: 322,069,808
Std  population: 4,158,441


## Question 2 – Best Year per series_id (max summed quarterly value)

In [5]:
# Keep only Q01-Q04 periods (exclude annual totals like Q05)
quarterly = bls_df[bls_df['period'].str.match(r'^Q0[1-4]$')].copy()

yearly_sum = (
    quarterly
    .groupby(['series_id', 'year'])['value']
    .sum()
    .reset_index(name='value')
)

# For each series pick the year with the highest sum
best_year = (
    yearly_sum
    .loc[yearly_sum.groupby('series_id')['value'].idxmax()]
    .reset_index(drop=True)
)

print(f'Unique series: {len(best_year)}')
best_year.head(10)


Unique series: 237


,series_id,year,value
0,PRS30006011,2022,16.400
1,PRS30006012,2022,13.000
2,PRS30006013,1998,564.713
3,PRS30006021,2010,14.200
4,PRS30006022,2010,8.900
5,PRS30006023,2014,402.512
6,PRS30006031,2022,16.500
7,PRS30006032,2021,13.800
8,PRS30006033,1998,561.703
9,PRS30006061,2022,27.600


## Question 3 – PRS30006032 / Q01 joined with Population

In [6]:
target = bls_df[
    (bls_df['series_id'] == 'PRS30006032') &
    (bls_df['period']    == 'Q01')
].copy()

# Population year column may be int or str – normalise
pop_lookup = pop_df[['Year', 'Population']].copy()
pop_lookup['Year'] = pop_lookup['Year'].astype(int)
target['year'] = target['year'].astype(int)

report = target.merge(
    pop_lookup,
    left_on='year',
    right_on='Year',
    how='left'
)[['series_id', 'year', 'period', 'value', 'Population']]

print(report.to_string(index=False))


  series_id  year period  value  Population
PRS30006032  1995    Q01    0.0         NaN
PRS30006032  1996    Q01   -4.2         NaN
PRS30006032  1997    Q01    2.8         NaN
PRS30006032  1998    Q01    0.9         NaN
PRS30006032  1999    Q01   -4.1         NaN
PRS30006032  2000    Q01    0.5         NaN
PRS30006032  2001    Q01   -6.3         NaN
PRS30006032  2002    Q01   -6.6         NaN
PRS30006032  2003    Q01   -5.7         NaN
PRS30006032  2004    Q01    2.0         NaN
PRS30006032  2005    Q01   -0.5         NaN
PRS30006032  2006    Q01    1.8         NaN
PRS30006032  2007    Q01   -0.8         NaN
PRS30006032  2008    Q01   -3.5         NaN
PRS30006032  2009    Q01  -21.0         NaN
PRS30006032  2010    Q01    3.2         NaN
PRS30006032  2011    Q01    1.5         NaN
PRS30006032  2012    Q01    2.5         NaN
PRS30006032  2013    Q01    0.5 316128839.0
PRS30006032  2014    Q01   -0.1 318857056.0
PRS30006032  2015    Q01   -1.7 321418821.0
PRS30006032  2016    Q01   -1.4 